In [59]:
!pip install fastapi uvicorn pillow opencv-python-headless python-multipart nest-asyncio requests deep-translator pyngrok -q

In [87]:
import requests
OCR_SPACE_API_KEY = "K87107613888957"

payload = {
    "url": "https://ocr.space/Content/Images/receipt-ocr-original.webp",
    "language": "eng"
}

response = requests.post(
    "https://api.ocr.space/parse/image",
    data=payload,
    headers={"apikey": OCR_SPACE_API_KEY}
)

print("HTTP Status:", response.status_code)
print(response.text)

if response.status_code == 200:
    data = response.json()
    print(data)

HTTP Status: 200
{"ParsedResults":[{"TextOverlay":{"Lines":[],"HasOverlay":false,"Message":"Text overlay is not provided as it is not requested"},"TextOrientation":"0","FileParseExitCode":1,"ParsedText":"Walmart\r\nSave money. Live better.\r\n( 330 ) 339 -\r\n3991\r\nMANAGER DIANA EARNEST\r\n231 BLUEBELL DR SW\r\nNEW PHILADELPHIA OH 44663\r\n02115 009044 44\r\n01301\r\nPET TOY\r\nFLOPPY PUPPY\r\nSSSUPREME S\r\nZ . 5 SQUEAK\r\nMUNCHY DMBEL\r\nDOG TREAT\r\nPED PCH 1\r\nPED PCH 1\r\nCOUPON 23100\r\nHNYMD SMORES\r\nFRENCH DRSNG\r\n3 ORANGES\r\nBABY CARROTS\r\nCOLLARDS\r\nCALZONE\r\nMM RVW MNT\r\nSTKOBRLPLABL\r\nSTKOBRLPLABL\r\nSTKO SUNFLWR\r\nSTKO SUNFLWR\r\nSTKO SUNFLWR\r\nSTKO SUNFLWR\r\nBLING BEADS\r\nGREAT VALUE\r\nLIPTON\r\nDRY DOG\r\nTAX\r\nUS DEBIT\r\n00474757165B\r\n004747514846\r\n070060332153\r\n084699803238\r\n068113108796\r\n007119013654\r\n002310011802\r\n002310011802\r\n052310037000\r\n088491226837\r\n004132100655\r\n001466835001\r\n003338366602\r\n1\r\n000000004614K1\r\n0052

In [88]:
import cv2
import numpy as np
from PIL import Image
import io
import base64

def preprocess_for_sign(image_bytes: bytes) -> bytes:
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    h, w = img.shape[:2]
    if w > 1800:
        scale = 1800 / w
        img = cv2.resize(
            img,
            (int(w * scale), int(h * scale)),
            interpolation=cv2.INTER_AREA
        )

    # Light enhancement only
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge([l, a, b])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    for quality in [90, 85, 80, 75]:
        _, encoded = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
        result_bytes = encoded.tobytes()

        size_kb = len(result_bytes) / 1024
        print(f"Trying quality={quality}, size={size_kb:.1f} KB")

        if size_kb < 900:  # target under 1MB
            return result_bytes

    return result_bytes  # fallback

In [89]:
from deep_translator import GoogleTranslator

def translate_to_arabic(text: str) -> str:
    try:
        translator = GoogleTranslator(source='en', target='ar')
        if len(text) <= 5000:
            return translator.translate(text)
        # Handle long text line by line
        lines = text.split('\n')
        translated_lines = []
        for line in lines:
            if line.strip():
                translated_lines.append(translator.translate(line.strip()))
            else:
                translated_lines.append("")
        return '\n'.join(translated_lines)
    except Exception as e:
        return f"Translation error: {e}"

def call_ocr_space(image_bytes: bytes, filename: str = "image.jpg") -> dict:
    processed_bytes = preprocess_for_sign(image_bytes)
    print("Original size KB:", len(image_bytes) / 1024)
    print("Processed size KB:", len(processed_bytes) / 1024)

    response = requests.post(
        "https://api.ocr.space/parse/image",
        files={"file": (filename, processed_bytes, "image/jpeg")},
        headers={"apikey": OCR_SPACE_API_KEY},
        data={
            "language": "eng",
            "OCREngine": 1,
            "scale": True,
            "detectOrientation": True,
            "isOverlayRequired": False
        }
    )

    print("OCR.space status:", response.status_code)
    print("OCR.space raw response:", response.text)

    try:
        result = response.json()
    except Exception as e:
        return {
            "success": False,
            "error": f"Failed to parse OCR.space JSON: {e}. Raw: {response.text[:500]}",
            "english_text": "",
            "arabic_text": "",
        }

    if result.get("IsErroredOnProcessing") is True:
        return {
            "success": False,
            "error": result.get("ErrorMessage", "OCR failed"),
            "english_text": "",
            "arabic_text": "",
        }

    parsed_results = result.get("ParsedResults")
    if not parsed_results:
        return {
            "success": False,
            "error": f"No ParsedResults returned. Full result: {result}",
            "english_text": "",
            "arabic_text": "",
        }

    english_text = parsed_results[0].get("ParsedText", "").strip()

    if not english_text:
        return {
            "success": False,
            "error": f"OCR returned empty text. Full result: {result}",
            "english_text": "",
            "arabic_text": "",
        }

    arabic_text = translate_to_arabic(english_text)

    return {
        "success": True,
        "english_text": english_text,
        "arabic_text": arabic_text,
        "is_rtl": True
    }

In [90]:
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import nest_asyncio
from pyngrok import ngrok

app = FastAPI(title="OCR API - Arabic & English Signs")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

@app.get("/")
def root():
    return {"status": "running", "supported_languages": ["Arabic", "English"]}

@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/ocr")
async def ocr_endpoint(file: UploadFile = File(...)):
    allowed = ["image/jpeg", "image/png", "image/webp", "image/jpg"]
    if file.content_type not in allowed:
        raise HTTPException(status_code=400,
                            detail="Only JPEG, PNG, WEBP allowed")

    image_bytes = await file.read()

    if len(image_bytes) > 5 * 1024 * 1024:
        raise HTTPException(status_code=400,
                            detail="Image too large. Max 5MB.")

    result = call_ocr_space(image_bytes, file.filename)

    if not result["success"]:
        raise HTTPException(status_code=502,
                            detail=result["error"])

    return result

print("FastAPI app ready ✓")

FastAPI app ready ✓


In [91]:
import nest_asyncio
import threading
import time
import uvicorn
from pyngrok import ngrok


nest_asyncio.apply()


ngrok.set_auth_token("3CBb3kuEuGDgXHbrdcpbguquCz2_4faYr1oF3SJGXyN3YsynA")

In [92]:
ngrok.kill()


In [93]:
nest_asyncio.apply()

ngrok.kill()

public_url = ngrok.connect(8000)

print("=" * 50)
print("Your public API URL:", public_url.public_url)
print("Test endpoint:", public_url.public_url + "/ocr")
print("Health check:", public_url.public_url + "/health")
print("=" * 50)

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()

Your public API URL: https://roundup-sleet-autopilot.ngrok-free.dev
Test endpoint: https://roundup-sleet-autopilot.ngrok-free.dev/ocr
Health check: https://roundup-sleet-autopilot.ngrok-free.dev/health


INFO:     Started server process [3087]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [94]:
response = requests.get("https://roundup-sleet-autopilot.ngrok-free.dev/health")
print(response.status_code)
print(response.text)

INFO:     34.86.8.69:0 - "GET /health HTTP/1.1" 200 OK
200
{"status":"ok"}


In [95]:
import os

size_bytes = os.path.getsize("IMG_0850.jpeg")
print("Bytes:", size_bytes)
print("KB:", size_bytes / 1024)
print("MB:", size_bytes / (1024 * 1024))

Bytes: 5168375
KB: 5047.2412109375
MB: 4.928946495056152


In [96]:
API_URL = "https://roundup-sleet-autopilot.ngrok-free.dev"

with open("IMG_0850.jpeg", "rb") as f:
    response = requests.post(
        f"{API_URL}/ocr",
        files={"file": ("IMG_0850.jpeg", f, "image/jpeg")}
    )

print("Status:", response.status_code)
print(response.text)

Trying quality=90, size=374.0 KB
Original size KB: 5047.2412109375
Processed size KB: 374.0048828125
OCR.space status: 200
OCR.space raw response: {"ParsedResults":[{"TextOverlay":{"Lines":[],"HasOverlay":false,"Message":"Text overlay is not provided as it is not requested"},"TextOrientation":"0","FileParseExitCode":1,"ParsedText":"FROM 1 TO 11 / 12/37\r\nPOST GRADUATE\r\nLAB RESEARCH COMP. LAB\r\nFROM 13 TO 20/ FROM 34 TO 36\r\nCOMP. APPLICATION\r\nLAB RESEARCH COMP. LAB\r\n","ErrorMessage":"","ErrorDetails":""}],"OCRExitCode":1,"IsErroredOnProcessing":false,"ProcessingTimeInMilliseconds":"3016","SearchablePDFURL":"Searchable PDF not generated as it was not requested."}
INFO:     34.86.8.69:0 - "POST /ocr HTTP/1.1" 200 OK
Status: 200
{"success":true,"english_text":"FROM 1 TO 11 / 12/37\r\nPOST GRADUATE\r\nLAB RESEARCH COMP. LAB\r\nFROM 13 TO 20/ FROM 34 TO 36\r\nCOMP. APPLICATION\r\nLAB RESEARCH COMP. LAB","arabic_text":"من 1 إلى 11 / 12/37\r\nما بعد التخرج\r\nشركات أبحاث المختبرات. م